# Credit Union Quarterly Data IngestionThis notebook executes the full-coverage ingestion pipeline in `ingest_ncua_call_report.py`.

In [0]:
# Configure your ingestion window (inclusive).# Example: start_year=1994 and end_year=2026 requests all quarters in that range.start_year = 1994end_year = 2026output_file = "NCUA_Call_Report.csv"multirecord_output_file = "NCUA_Call_Report_Multirecord.csv"run_ingestion(    start_year=start_year,    end_year=end_year,    output_file=output_file,    multirecord_output_file=multirecord_output_file,    cleanup_temp_files=True,)

In [0]:
from ingest_ncua_call_report import run_ingestion
from pyspark.sql.functions import to_date, col
import os
import re
import gc

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
START_YEAR = 1994
END_YEAR = 2026
SCHEMA = "workspace.credituniontest"
MAIN_TABLE = f"{SCHEMA}.ncua_call_report"
MULTI_TABLE = f"{SCHEMA}.ncua_call_report_multirecord"
MAPPING_TABLE = f"{SCHEMA}.ncua_column_mapping"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def sanitize_column_name(header: str) -> tuple[str, str]:
    """Extract account code and description from 'CODE - Description' format."""
    parts = header.split(" - ", maxsplit=1)
    code = parts[0].strip()
    description = parts[1].strip() if len(parts) > 1 else ""
    clean = re.sub(r"[^a-zA-Z0-9_]", "_", code)
    clean = re.sub(r"_+", "_", clean).strip("_").upper()
    return clean, description


def sanitize_df_columns(df, column_mapping_accum):
    """Rename all columns on a Spark DataFrame, deduplicate, track mapping."""
    seen = {}
    for original in df.columns:
        clean, desc = sanitize_column_name(original)
        if clean in seen:
            seen[clean] += 1
            clean = f"{clean}_{seen[clean]}"
        else:
            seen[clean] = 0
        column_mapping_accum.append((original, clean, desc))
        df = df.withColumnRenamed(original, clean)
    return df


def add_report_date(df):
    """Parse CYCLE_DATE (numeric YYYYMMDD) into a proper DATE column."""
    if "CYCLE_DATE" in df.columns:
        df = df.withColumn(
            "report_date",
            to_date(col("CYCLE_DATE").cast("string"), "yyyyMMdd"),
        )
    return df


def cleanup_files(*paths):
    for p in paths:
        try:
            if os.path.exists(p):
                os.remove(p)
        except OSError:
            pass

# ---------------------------------------------------------------------------
# Year-by-year ingestion loop
# ---------------------------------------------------------------------------
all_column_mappings = []
first_write = True
years_ok = []
years_failed = []

for year in range(START_YEAR, END_YEAR + 1):
    output_csv = f"ncua_main_{year}.csv"
    multi_csv = f"ncua_multi_{year}.csv"

    print(f"\n{'='*60}")
    print(f"Processing year {year}")
    print(f"{'='*60}")

    try:
        run_ingestion(
            start_year=year,
            end_year=year,
            output_file=output_csv,
            multirecord_output_file=multi_csv,
            cleanup_temp_files=True,
        )

        # -- Main table --
        if os.path.exists(output_csv) and os.path.getsize(output_csv) > 0:
            df = spark.read.csv(output_csv, header=True, inferSchema=True)
            df = sanitize_df_columns(df, all_column_mappings)
            df = add_report_date(df)

            write_mode = "overwrite" if first_write else "append"
            (
                df.write
                .mode(write_mode)
                .option("mergeSchema", "true")
                .saveAsTable(MAIN_TABLE)
            )
            row_count = df.count()
            print(f"  -> {row_count:,} rows written to {MAIN_TABLE} ({write_mode})")
            first_write = False
            del df
        else:
            print(f"  -> No main data for {year}, skipping.")

        # -- Multirecord table --
        if os.path.exists(multi_csv) and os.path.getsize(multi_csv) > 0:
            multi_df = spark.read.csv(multi_csv, header=True, inferSchema=True)
            multi_df = sanitize_df_columns(multi_df, all_column_mappings)
            multi_df = add_report_date(multi_df)
            multi_df.write.mode("append").option("mergeSchema", "true").saveAsTable(MULTI_TABLE)
            print(f"  -> Multirecord rows written to {MULTI_TABLE}")
            del multi_df

        years_ok.append(year)

    except Exception as e:
        print(f"  -> FAILED: {type(e).__name__}: {e}")
        years_failed.append((year, str(e)))

    finally:
        cleanup_files(output_csv, multi_csv)
        gc.collect()

# ---------------------------------------------------------------------------
# Column mapping reference table (deduplicated)
# ---------------------------------------------------------------------------
if all_column_mappings:
    seen_mappings = {}
    deduped = []
    for orig, clean, desc in all_column_mappings:
        if clean not in seen_mappings:
            seen_mappings[clean] = True
            deduped.append((orig, clean, desc))
    mapping_df = spark.createDataFrame(deduped, ["original_header", "column_name", "description"])
    mapping_df.write.mode("overwrite").saveAsTable(MAPPING_TABLE)
    print(f"\nColumn mapping saved to {MAPPING_TABLE} ({len(deduped)} entries)")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"COMPLETE: {len(years_ok)} years succeeded, {len(years_failed)} failed")
if years_failed:
    print("Failed years:")
    for y, reason in years_failed:
        print(f"  {y}: {reason}")

total = spark.table(MAIN_TABLE).count()
cols = len(spark.table(MAIN_TABLE).columns)
print(f"Final table: {total:,} rows x {cols} columns in {MAIN_TABLE}")